# 2-Class Hand X-Ray Validator Training (Real Data Only)

Trains `HandXRayValidator` (ImageNet-pretrained `ResNet-18`, 2 classes: `["valid_pediatric_hand_xray", "other"]`) on **real data only**.

### Files to Upload to Colab (or place in Google Drive)
1. **`rsna_positives.zip`** — Real RSNA hand/wrist X-ray PNGs (e.g. 500–2,000 images from `BoneAgePrediction-1/data/boneage-training-dataset/boneage-training-dataset`).
2. **`validator_negatives.zip`** — Real negative images (non-hand X-rays such as chest/skull/pelvis/knee, natural photos, animals, documents, screenshots).
3. *(Optional)* **`backend/app/models/validator.py`** — Uploaded to verify byte-for-byte parity with the shared `preprocess_validator_image` and `HandXRayValidator` definition.

### Output Produced
- `hand_validator_best.pt` — Download and place into `C:\bonetest\backend\checkpoints\hand_validator_best.pt`.

In [ ]:
# 1. Check GPU & Unpack Uploaded Zips
import os, zipfile
from pathlib import Path
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

POS_DIR = Path('/content/data/positives')
NEG_DIR = Path('/content/data/negatives')
POS_DIR.mkdir(parents=True, exist_ok=True)
NEG_DIR.mkdir(parents=True, exist_ok=True)

if Path('/content/rsna_positives.zip').exists():
    with zipfile.ZipFile('/content/rsna_positives.zip', 'r') as z:
        z.extractall(POS_DIR)
if Path('/content/validator_negatives.zip').exists():
    with zipfile.ZipFile('/content/validator_negatives.zip', 'r') as z:
        z.extractall(NEG_DIR)


In [ ]:
# 2. Shared Model Architecture & Preprocessing (Exact Match with backend/app/models/validator.py)
from typing import List
import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models

VALIDATOR_CLASSES: List[str] = [
    'valid_pediatric_hand_xray',
    'other',
]

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def letterbox_rgb_224(image_rgb: np.ndarray, target_size: int = 224) -> np.ndarray:
    h, w = image_rgb.shape[:2]
    scale = target_size / float(max(h, w))
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    resized = cv2.resize(image_rgb, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((target_size, target_size, 3), dtype=np.uint8)
    y_off = (target_size - new_h) // 2
    x_off = (target_size - new_w) // 2
    canvas[y_off : y_off + new_h, x_off : x_off + new_w] = resized
    return canvas

def preprocess_validator_image(image_bgr: np.ndarray, target_size: int = 224) -> torch.Tensor:
    if image_bgr.ndim == 2:
        rgb = cv2.cvtColor(image_bgr, cv2.COLOR_GRAY2RGB)
    elif image_bgr.shape[2] == 4:
        rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGRA2RGB)
    else:
        rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    letterboxed = letterbox_rgb_224(rgb, target_size=target_size)
    norm = letterboxed.astype(np.float32) / 255.0
    norm = (norm - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(norm.transpose(2, 0, 1)).float()

class HandXRayValidator(nn.Module):
    def __init__(self, num_classes: int = len(VALIDATOR_CLASSES), dropout_rate: float = 0.3, pretrained: bool = False) -> None:
        super().__init__()
        self.num_classes = num_classes
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),
            nn.Linear(128, num_classes),
        )

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.extract_features(x))

    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        return torch.softmax(self.forward(x), dim=-1)


In [ ]:
# 3. Build Real Dataset Splits & Train on GPU
import random
from torch.utils.data import Dataset, DataLoader

EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.webp', '.tif', '.tiff'}
pos_files = sorted([p for p in POS_DIR.rglob('*') if p.is_file() and p.suffix.lower() in EXTS])
neg_files = sorted([p for p in NEG_DIR.rglob('*') if p.is_file() and p.suffix.lower() in EXTS])
print(f'Found {len(pos_files)} real positive RSNA images and {len(neg_files)} real negative images.')
assert len(pos_files) > 0 and len(neg_files) > 0, 'Both real positive and real negative images are required!'

rng = random.Random(42)
rng.shuffle(pos_files)
rng.shuffle(neg_files)

def split_70_15_15(items):
    n = len(items)
    n_tr, n_val = int(n * 0.70), int(n * 0.15)
    return items[:n_tr], items[n_tr:n_tr+n_val], items[n_tr+n_val:]

p_tr, p_val, p_te = split_70_15_15([(p, 0) for p in pos_files])
n_tr, n_val, n_te = split_70_15_15([(p, 1) for p in neg_files])
train_items, val_items, test_items = p_tr + n_tr, p_val + n_val, p_te + n_te
rng.shuffle(train_items)

class DiskDataset(Dataset):
    def __init__(self, items, augment=False):
        self.items = items
        self.augment = augment
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if self.augment and random.random() < 0.5:
            img = cv2.flip(img, 1)
        return preprocess_validator_image(img, 224), torch.tensor(label, dtype=torch.long)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader = DataLoader(DiskDataset(train_items, augment=True), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(DiskDataset(val_items, augment=False), batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(DiskDataset(test_items, augment=False), batch_size=32, shuffle=False, num_workers=2)

model = HandXRayValidator(num_classes=2, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

best_val_loss = float('inf')
for epoch in range(1, 11):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    scheduler.step()

    model.eval()
    vl_loss, vl_corr, vl_tot = 0.0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            vl_loss += criterion(out, y).item() * x.size(0)
            vl_corr += (out.argmax(1) == y).sum().item()
            vl_tot += x.size(0)
    vl_loss /= max(1, vl_tot)
    vl_acc = vl_corr / max(1, vl_tot)
    print(f'Epoch {epoch:02d}/10 | Val Loss: {vl_loss:.4f} | Val Acc: {vl_acc*100:.2f}%')
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'classes': list(VALIDATOR_CLASSES),
            'val_loss': vl_loss,
            'val_acc': vl_acc,
            'epoch': epoch,
            'model_architecture': 'ResNet18-2Class-Validator',
        }, '/content/hand_validator_best.pt')


In [ ]:
# 4. Held-Out ROC, RSNA Accept Rate, False-Accept Rate & Download Checkpoint
from sklearn.metrics import roc_auc_score, roc_curve

ckpt = torch.load('/content/hand_validator_best.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'], strict=True)
model.eval()

y_true_pos, p_hand = [], []
with torch.no_grad():
    for x, y in test_loader:
        probs = model.predict_proba(x.to(device)).cpu().numpy()
        p_hand.extend(probs[:, 0].tolist())
        y_true_pos.extend((y.numpy() == 0).astype(int).tolist())

y_true_pos = np.array(y_true_pos)
p_hand = np.array(p_hand)
auc = roc_auc_score(y_true_pos, p_hand)
print(f'Held-Out Test ROC AUC: {auc:.4f}')
for thr in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    accept_pos = np.mean(p_hand[y_true_pos == 1] >= thr) * 100
    false_accept_neg = np.mean(p_hand[y_true_pos == 0] >= thr) * 100
    print(f'Threshold={thr:.2f} | RSNA Accept Rate={accept_pos:6.2f}% | False-Accept Rate={false_accept_neg:6.2f}%')
